# D1.3 · Learning systems, text, recommendations and speech

## What are we doing?

We will explore several kinds of AI systems using tiny examples. Each part shows the input, the operation and the result so learners can see what the system is doing.

Run one cell at a time with Shift+Enter. Read the explanation before running each cell.

## Setup

Load the tools used by these examples.

In [ ]:
import numpy as np
import pandas as pd

## 1. Rank ideas by readiness

**What and why.** We score value, available data and difficulty. This helps us choose an achievable first project.

**Look for.** B scores 13 and A scores 10. Discuss what the scores leave out, including risk.

In [ ]:
ideas = pd.DataFrame({'idea': ['A', 'B'], 'value': [5, 3], 'data': [2, 5], 'feasibility': [3, 5]})
ideas['total'] = ideas['value'] + ideas['data'] + ideas['feasibility']
print(ideas.sort_values('total', ascending=False))

## 2. Mask personal information

**What and why.** We replace an email before processing a text. This demonstrates data minimisation.

**Look for.** The email is replaced by [EMAIL].

In [ ]:
import re
text = 'Please reply to sam@example.com'
masked = re.sub(r'[\w.]+@[\w.]+', '[EMAIL]', text)
print(masked)

## 3. Words, stop words and pairs

**What and why.** We split text into words, remove a common word and list adjacent word pairs.

**Look for.** The word 'the' is removed and adjacent pairs remain.

In [ ]:
words = 'the card is blocked'.split()
useful = [word for word in words if word != 'the']
print('Words:', words)
print('Without the:', useful)
print('Word pairs:', list(zip(useful, useful[1:])))

## 4. Counts and TF-IDF

**What and why.** We turn words into numeric columns so a model can use text.

**Look for.** The columns represent words. TF-IDF assigns weights based on their occurrence.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
texts = ['card fee', 'card problem', 'loan fee']
counts = CountVectorizer()
print('Words:', counts.fit(texts).get_feature_names_out())
print('Counts:\n', counts.transform(texts).toarray())
tfidf = TfidfVectorizer()
print('TF-IDF:\n', tfidf.fit_transform(texts).toarray().round(2))

## 5. Classify short texts

**What and why.** We train a small classifier to show the input and output of complaint routing.

**Look for.** The model predicts a class. Tiny training data is for demonstration only.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
texts = ['card blocked', 'card failed', 'loan payment', 'loan term']
labels = ['card', 'card', 'loan', 'loan']
model = make_pipeline(CountVectorizer(), LogisticRegression())
model.fit(texts, labels)
print('Predicted label:', model.predict(['card problem'])[0])

## 6. Compare with a baseline

**What and why.** A baseline always chooses the most common label. We compare it with a prediction on a small check set.

**Look for.** The baseline is 0.50 and the example is 0.75 on these four rows.

In [ ]:
from sklearn.metrics import accuracy_score
actual = ['card', 'loan', 'card', 'loan']
baseline = ['card', 'card', 'card', 'card']
example = ['card', 'loan', 'card', 'card']
print('Baseline accuracy:', accuracy_score(actual, baseline))
print('Example accuracy:', accuracy_score(actual, example))

## 7. Explain words, identify a number, and escalate

**What and why.** We identify an amount and flag a word for human review. Real systems require wider testing.

**Look for.** The amount is 25 and urgent is flagged for review.

In [ ]:
import re
message = 'urgent fee 25'
print('Amount:', re.findall(r'\d+', message))
print('Human review:', 'urgent' in message)
print('Helpful words:', message.split())

## 8. Support, confidence and lift

**What and why.** We count items bought together. Lift compares a conditional rate with the overall rate.

**Look for.** Support is 0.375, confidence 0.75 and lift 1.5.

In [ ]:
a = np.array([1, 1, 1, 1, 0, 0, 0, 0])
b = np.array([1, 1, 1, 0, 1, 0, 0, 0])
support = ((a == 1) & (b == 1)).mean()
confidence = b[a == 1].mean()
lift = confidence / b.mean()
print('Support:', support, 'Confidence:', confidence, 'Lift:', lift)

## 9. Recommend from similar choices

**What and why.** We use a shared selection to suggest an item another person chose.

**Look for.** Both chose A, so C is a simple suggestion for person 1.

In [ ]:
choices = {'person 1': {'A', 'B'}, 'person 2': {'A', 'C'}}
print('Shared:', choices['person 1'] & choices['person 2'])
print('Suggested to person 1:', choices['person 2'] - choices['person 1'])

## 10. Cold start and coverage

**What and why.** A new person has no history. We can begin with popular choices and count how many different items we show.

**Look for.** The fallback is A and coverage is three different items.

In [ ]:
suggestions = ['A', 'B', 'A', 'C']
print('Fallback for new person: A')
print('Different items shown:', len(set(suggestions)))

## 11. Sampling and frequency

**What and why.** Sound is represented by samples. The Nyquist limit is half the sampling rate.

**Look for.** 8,000 samples per second gives a 4,000 Hz Nyquist limit.

In [ ]:
sample_rate = 8000
print('Samples per second:', sample_rate)
print('Highest represented frequency:', sample_rate / 2, 'Hz')

## 12. Waveform and frequencies

**What and why.** We construct a simple wave and look at the strongest frequency.

**Look for.** The strongest frequency is 5 Hz.

In [ ]:
rate = 100
time = np.arange(100) / rate
wave = np.sin(2 * np.pi * 5 * time)
frequencies = np.fft.rfftfreq(len(wave), 1 / rate)
strength = np.abs(np.fft.rfft(wave))
print('Strongest frequency:', frequencies[strength.argmax()], 'Hz')

## 13. Spectrogram and telephone keypad tones

**What and why.** A spectrogram shows how frequencies change over time. Keypad tones use two frequencies together.

**Look for.** The strongest frequency bins are close to 697 Hz and 1209 Hz. A spectrogram stores frequency strength for each time window.

In [ ]:
from scipy.signal import spectrogram
rate = 8000
time = np.arange(800) / rate
tone = np.sin(2*np.pi*697*time) + np.sin(2*np.pi*1209*time)
frequencies, times, power = spectrogram(tone, fs=rate, nperseg=256)
print('Frequency bins:', len(frequencies), 'Time windows:', len(times))
strong = np.argsort(power.mean(axis=1))[-2:]
print('Strongest frequencies (approximately):', frequencies[strong])

## 14. Speech features and noise

**What and why.** We compare the average magnitude of a simple wave before and after adding noise. Speech systems use richer features such as MFCCs.

**Look for.** Noise changes the values. MFCCs summarise frequency patterns in speech.

In [ ]:
rng = np.random.default_rng(0)
wave = np.sin(np.linspace(0, 6, 100))
noisy = wave + rng.normal(0, 0.5, 100)
print('Clean average magnitude:', round(np.abs(wave).mean(), 2))
print('Noisy average magnitude:', round(np.abs(noisy).mean(), 2))

## 15. Recommendation hit rate at two

**What and why.** We check whether the actual choice appears among the first two suggestions.

**Look for.** The actual choice is present and one of two suggestions is a hit.

In [ ]:
suggested = ['A', 'B']
actual = 'B'
print('Hit in top two:', actual in suggested)
print('Precision at two:', int(actual in suggested) / len(suggested))

## 16. Word error rate and downstream errors

**What and why.** We count one substituted word. An incorrect transcript can also change a later text classification.

**Look for.** One substitution among three reference words gives a word error rate of 0.33.

In [ ]:
reference = 'show my balance'
transcript = 'show my challenge'
substitutions = 1
wer = substitutions / len(reference.split())
print('Reference:', reference)
print('Transcript:', transcript)
print('Word error rate:', round(wer, 2))